# **1. Descripción del Problema**

**Problema:** Clasificación binaria de reseñas de Amazon como positivas (1) o negativas (0)

**Objetivo:** Predecir el sentimiento a partir del texto de la reseña

# **2. Carga y Exploración Inicial (EDA)**

In [28]:
import kagglehub

# Descarga la última versión del dataset de Kaggle
path = kagglehub.dataset_download("bittlingmayer/amazonreviews")

# Imprime la ruta donde se encuentran descargados los archivos del dataset
print("Path to dataset files:", path)

Path to dataset files: C:\Users\salom\.cache\kagglehub\datasets\bittlingmayer\amazonreviews\versions\7


In [29]:
# Como Kagglehub descarga los archivos en una carpeta oculta dentro del caché de usuario, 
# vamos a copiarlos a una ubicación más accesible para su análisis.

import shutil
import os

# Ruta donde Kagglehub guarda los archivos descargados
origen_path = r"C:\Users\salom\.cache\kagglehub\datasets\bittlingmayer\amazonreviews\versions\7"

# Ruta destino donde queremos copiar los archivos
destino_path = r"C:\Users\salom\Desktop\2025-2\Análisis de Datos\Momento Evaluativo 2\Analisis_de_datos\Evaluación 4\Dataset Amazon Reviews Dataset"

# Copiar todos los archivos del origen al destino
for archivo in os.listdir(origen_path):
    ruta_origen = os.path.join(origen_path, archivo)
    ruta_destino = os.path.join(destino_path, archivo)
    if os.path.isfile(ruta_origen):
        shutil.copy2(ruta_origen, ruta_destino)

print("Archivos copiados exitosamente a:", destino_path)


Archivos copiados exitosamente a: C:\Users\salom\Desktop\2025-2\Análisis de Datos\Momento Evaluativo 2\Analisis_de_datos\Evaluación 4\Dataset Amazon Reviews Dataset


In [32]:
# Ahora procedemos con el análisis de datos utilizando pandas, numpy, matplotlib, seaborn, y nltk.

import bz2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Descargar recursos NLTK 
nltk.download('stopwords') # Se usa para eliminar palabras comunes como "y", "el", "de", etc.
nltk.download('punkt') # Se usa para tokenizar el texto en palabras como "Hola, ¿cómo estás?" -> ["Hola", ",", "¿", "cómo", "estás", "?"]

# Función para leer y parsear archivos .bz2
def parse_bz2_file(file_path, sample_size=50000):

    # Listas para almacenar los datos
    reviews = []
    labels = []
    
    # En el ejercicio inicial se usó un enfoque incorrecto para abrir archivos bz2, por lo tanto, con el try-except se maneja cualquier error.
    try:
        print(f"Intentando abrir: {file_path}")
        print("Archivo abierto correctamente.")
        
        # Abrir el archivo bz2 correctamente
        with bz2.open(file_path, 'rt', encoding='utf-8') as file:
            for i, line in enumerate(file):
                if i >= sample_size:
                    break
                    
                # Mostrar progreso cada 10,000 líneas para monitorear el avance
                if i % 10000 == 0 and i > 0:
                    print(f"Procesadas {i} líneas...")
                
                # Parsear la línea (formato: __label__1 o __label__2)
                if line.startswith('__label__1'):
                    labels.append(1)  # Positivo
                    reviews.append(line[len('__label__1 '):].strip())
                elif line.startswith('__label__2'):
                    labels.append(0)  # Negativo
                    reviews.append(line[len('__label__2 '):].strip())
        
        # Confirmar el número de reseñas procesadas
        print(f"Procesamiento completado: {len(reviews)} reseñas")
        print("\n")
        
    except Exception as e:
        print(f"Error: {e}")
        return None
    
    # Crear DataFrame
    return pd.DataFrame({'review_text': reviews, 'sentiment': labels})

# Mensaje para indicar que se están cargando los datos y que el proceso hasta ahora es correcto
print("\n")
print("=== CARGANDO DATOS CON RUTAS CORRECTAS ===")
print("\n")

# Definir la ruta base donde se encuentran los archivos
base_path = r"C:\Users\salom\Desktop\2025-2\Análisis de Datos\Momento Evaluativo 2\Analisis_de_datos\Evaluación 4\Dataset Amazon Reviews Dataset"

# Rutas completas a los archivos de entrenamiento y prueba
train_path = os.path.join(base_path, "train.ft.txt.bz2")
test_path = os.path.join(base_path, "test.ft.txt.bz2")

# Cargar dataset de entrenamiento
df_train = parse_bz2_file(train_path, sample_size=50000)

# Cargar dataset de prueba  
df_test = parse_bz2_file(test_path, sample_size=10000)

# Verificar que se cargó correctamente el dataset de entrenamiento
if df_train is not None:
    print(f"Distribución de sentimientos en train:")
    print(df_train['sentiment'].value_counts())
else:
    print("Error cargando train")

# Verificar que se cargó correctamente el dataset de prueba
if df_test is not None:
    print(f"Distribución de sentimientos en test:")
    print(df_test['sentiment'].value_counts())
else:
    print("Error cargando test")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\salom\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\salom\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!




=== CARGANDO DATOS CON RUTAS CORRECTAS ===


Intentando abrir: C:\Users\salom\Desktop\2025-2\Análisis de Datos\Momento Evaluativo 2\Analisis_de_datos\Evaluación 4\Dataset Amazon Reviews Dataset\train.ft.txt.bz2
Archivo abierto correctamente.
Procesadas 10000 líneas...
Procesadas 20000 líneas...
Procesadas 30000 líneas...
Procesadas 40000 líneas...
Procesamiento completado: 50000 reseñas


Intentando abrir: C:\Users\salom\Desktop\2025-2\Análisis de Datos\Momento Evaluativo 2\Analisis_de_datos\Evaluación 4\Dataset Amazon Reviews Dataset\test.ft.txt.bz2
Archivo abierto correctamente.
Procesamiento completado: 10000 reseñas


Distribución de sentimientos en train:
sentiment
0    25506
1    24494
Name: count, dtype: int64
Distribución de sentimientos en test:
sentiment
0    5125
1    4875
Name: count, dtype: int64
